# 🏎️ Partie 3 — Transformations avancées & score composite pilote

**Responsable : Salma**

Objectif : Construire une table analytique enrichie à partir des données F1 nettoyées, créer des indicateurs avancés avec PySpark, puis élaborer un score composite permettant de comparer les pilotes de manière plus équitable.

Dans cette partie, nous allons dépasser les statistiques descriptives brutes de la partie 2.  
L’objectif est de construire un score pilote qui tienne compte non seulement des résultats en course, mais aussi de la régularité, de la domination intra-équipe et de la performance relative par saison.


In [20]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [21]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

## 1. Initialisation de l’environnement PySpark

Dans cette partie, nous utilisons PySpark pour construire une table analytique enrichie et créer des indicateurs avancés à partir des données nettoyées en Partie 1.

In [38]:
spark = SparkSession.builder \
    .appName("F1_Transformations_Avancees_Score_Pilote") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark version : {spark.version}")

✅ Spark version : 4.1.1


## 2. Chargement des données nettoyées

Nous travaillons ici à partir des fichiers nettoyés produits dans la partie 1.  
Cela permet de séparer clairement la phase de préparation des données et la phase de transformation analytique.

In [23]:
DATA_PATH = "../data/processed/"

results = spark.read.csv(DATA_PATH + "results_clean.csv", header=True, inferSchema=True)
races = spark.read.csv(DATA_PATH + "races_clean.csv", header=True, inferSchema=True)
drivers = spark.read.csv(DATA_PATH + "drivers_clean.csv", header=True, inferSchema=True)
constructors = spark.read.csv(DATA_PATH + "constructors_clean.csv", header=True, inferSchema=True)
driver_standings = spark.read.csv(DATA_PATH + "driver_standings_clean.csv", header=True, inferSchema=True)
constructor_standings = spark.read.csv(DATA_PATH + "constructor_standings_clean.csv", header=True, inferSchema=True)

print("📦 Données chargées :")
for name, df in [
    ("results", results),
    ("races", races),
    ("drivers", drivers),
    ("constructors", constructors),
    ("driver_standings", driver_standings),
    ("constructor_standings", constructor_standings),
]:
    print(f"{name:<25} → {df.count():>6} lignes | {len(df.columns)} colonnes")

📦 Données chargées :
results                   →  26759 lignes | 20 colonnes
races                     →   1125 lignes | 8 colonnes
drivers                   →    861 lignes | 10 colonnes
constructors              →    212 lignes | 5 colonnes
driver_standings          →  34863 lignes | 7 colonnes
constructor_standings     →  13391 lignes | 7 colonnes


In [24]:
results.printSchema()
races.printSchema()
drivers.printSchema()
constructors.printSchema()

root
 |-- resultId: integer (nullable = true)
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- constructorId: integer (nullable = true)
 |-- number: integer (nullable = true)
 |-- grid: integer (nullable = true)
 |-- position: integer (nullable = true)
 |-- positionText: string (nullable = true)
 |-- positionOrder: integer (nullable = true)
 |-- points: double (nullable = true)
 |-- laps: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: double (nullable = true)
 |-- fastestLap: integer (nullable = true)
 |-- rank: integer (nullable = true)
 |-- fastestLapTime: string (nullable = true)
 |-- fastestLapSpeed: double (nullable = true)
 |-- statusId: integer (nullable = true)
 |-- position_int: double (nullable = true)
 |-- finished: boolean (nullable = true)

root
 |-- raceId: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- round: integer (nullable = true)
 |-- circuitId: integer (nullable = true)
 |-- 

## 3. Construction d’une table analytique enrichie

La table `results` constitue la base de notre analyse, car elle contient les résultats de chaque pilote pour chaque course.

Nous l’enrichissons en y ajoutant :
- les informations de course (`races`)
- les informations pilote (`drivers`)
- les informations écurie (`constructors`)

On obtient ainsi une table analytique au niveau :
**1 ligne = 1 pilote dans 1 course**

In [39]:
df_f1 = (
    results
    .join(
        races.select("raceId", "year", "round", "name", "date"),
        on="raceId",
        how="left"
    )
    .join(
        drivers.select("driverId", "fullName", "nationality"),
        on="driverId",
        how="left"
    )
    .join(
        constructors.select("constructorId", F.col("name").alias("constructor_name")),
        on="constructorId",
        how="left"
    )
)

print("✅ Table analytique enrichie créée")
print(f"Lignes : {df_f1.count()} | Colonnes : {len(df_f1.columns)}")
df_f1.show(5, truncate=False)


df_f1.cache()
_ = df_f1.count()
print("✅ Table enrichie mise en cache")

✅ Table analytique enrichie créée
Lignes : 26759 | Colonnes : 27
+-------------+--------+------+--------+------+----+--------+------------+-------------+------+----+----+------------+----------+----+--------------+---------------+--------+------------+--------+----+-----+---------------------+----------+-----------------+-----------+----------------+
|constructorId|driverId|raceId|resultId|number|grid|position|positionText|positionOrder|points|laps|time|milliseconds|fastestLap|rank|fastestLapTime|fastestLapSpeed|statusId|position_int|finished|year|round|name                 |date      |fullName         |nationality|constructor_name|
+-------------+--------+------+--------+------+----+--------+------------+-------------+------+----+----+------------+----------+----+--------------+---------------+--------+------------+--------+----+-----+---------------------+----------+-----------------+-----------+----------------+
|4            |12      |18    |12      |6     |20  |NULL    |R         

## 4. Création des indicateurs de performance

Nous construisons ici plusieurs variables dérivées permettant de mesurer la performance d’un pilote sur une course donnée :

- victoire
- podium
- top 10
- course terminée
- points marqués
- places gagnées entre la grille de départ et l’arrivée

Ces indicateurs serviront ensuite à construire un score composite.

In [26]:
df_perf = (
    df_f1
    .withColumn("win_flag", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("podium_flag", F.when(F.col("positionOrder") <= 3, 1).otherwise(0))
    .withColumn("top10_flag", F.when(F.col("positionOrder") <= 10, 1).otherwise(0))
    .withColumn("points_flag", F.when(F.col("points") > 0, 1).otherwise(0))
    .withColumn("finish_flag", F.when(F.col("finished") == True, 1).otherwise(0))
    .withColumn(
        "positions_gained",
        F.when(
            F.col("grid").isNotNull() & F.col("positionOrder").isNotNull(),
            F.col("grid") - F.col("positionOrder")
        ).otherwise(None)
    )
)

df_perf.select(
    "driverId", "fullName", "year", "round",
    "grid", "positionOrder", "points",
    "win_flag", "podium_flag", "top10_flag",
    "points_flag", "finish_flag", "positions_gained"
).show(10, truncate=False)

+--------+-----------------+----+-----+----+-------------+------+--------+-----------+----------+-----------+-----------+----------------+
|driverId|fullName         |year|round|grid|positionOrder|points|win_flag|podium_flag|top10_flag|points_flag|finish_flag|positions_gained|
+--------+-----------------+----+-----+----+-------------+------+--------+-----------+----------+-----------+-----------+----------------+
|12      |Nelson Piquet Jr.|2008|1    |20  |12           |0.0   |0       |0          |0         |0          |0          |8               |
|13      |Felipe Massa     |2008|1    |4   |13           |0.0   |0       |0          |0         |0          |0          |-9              |
|14      |David Coulthard  |2008|1    |8   |14           |0.0   |0       |0          |0         |0          |0          |-6              |
|18      |Jenson Button    |2008|1    |12  |18           |0.0   |0       |0          |0         |0          |0          |-6              |
|11      |Takuma Sato      

## 5. Comparaison intra-équipe avec window functions

L’un des principaux biais lorsqu’on compare les pilotes de F1 est la qualité de la voiture.  
Pour réduire cet effet, nous comparons les pilotes à leur coéquipier dans une même course et une même écurie.

Cette logique est particulièrement pertinente car les pilotes courent dans le même contexte, ils disposent d’une voiture comparable, et ils affrontent la même course.

Nous utilisons ici des **window functions** pour classer les pilotes à l’intérieur de chaque écurie et de chaque Grand Prix.

In [27]:
w_team_race = Window.partitionBy("raceId", "constructorId").orderBy(F.col("positionOrder").asc_nulls_last())

In [28]:
df_team = (
    df_perf
    .withColumn("team_finish_rank", F.row_number().over(w_team_race))
    .withColumn("beat_teammate_flag", F.when(F.col("team_finish_rank") == 1, 1).otherwise(0))
)

df_team.select(
    "raceId", "constructorId", "constructor_name",
    "driverId", "fullName", "positionOrder",
    "team_finish_rank", "beat_teammate_flag"
).orderBy("raceId", "constructorId", "team_finish_rank").show(20, truncate=False)

+------+-------------+----------------+--------+--------------------+-------------+----------------+------------------+
|raceId|constructorId|constructor_name|driverId|fullName            |positionOrder|team_finish_rank|beat_teammate_flag|
+------+-------------+----------------+--------+--------------------+-------------+----------------+------------------+
|1     |1            |McLaren         |5       |Heikki Kovalainen   |19           |1               |1                 |
|1     |1            |McLaren         |1       |Lewis Hamilton      |20           |2               |0                 |
|1     |2            |BMW Sauber      |2       |Nick Heidfeld       |10           |1               |1                 |
|1     |2            |BMW Sauber      |9       |Robert Kubica       |14           |2               |0                 |
|1     |3            |Williams        |3       |Nico Rosberg        |6            |1               |1                 |
|1     |3            |Williams        |6

Nous ajoutons également la part de points marqués dans l’écurie.  
Cela permet d’identifier si un pilote porte l’essentiel de la performance de son équipe.

In [29]:
w_team_points = Window.partitionBy("raceId", "constructorId")

df_team = (
    df_team
    .withColumn("team_total_points_race", F.sum("points").over(w_team_points))
    .withColumn(
        "team_points_share",
        F.when(F.col("team_total_points_race") > 0,
               F.col("points") / F.col("team_total_points_race"))
         .otherwise(0)
    )
)

df_team.select(
    "raceId", "constructor_name", "fullName",
    "points", "team_total_points_race", "team_points_share"
).show(15, truncate=False)

+------+----------------+------------------+------+----------------------+-------------------+
|raceId|constructor_name|fullName          |points|team_total_points_race|team_points_share  |
+------+----------------+------------------+------+----------------------+-------------------+
|1     |McLaren         |Heikki Kovalainen |0.0   |0.0                   |0.0                |
|1     |McLaren         |Lewis Hamilton    |0.0   |0.0                   |0.0                |
|1     |BMW Sauber      |Robert Kubica     |0.0   |0.0                   |0.0                |
|1     |BMW Sauber      |Nick Heidfeld     |0.0   |0.0                   |0.0                |
|1     |Williams        |Kazuki Nakajima   |0.0   |3.0                   |0.0                |
|1     |Williams        |Nico Rosberg      |3.0   |3.0                   |1.0                |
|1     |Renault         |Fernando Alonso   |4.0   |4.0                   |1.0                |
|1     |Renault         |Nelson Piquet Jr. |0.0   

## 6. Agrégation par pilote et par saison

Pour comparer les pilotes de manière plus juste entre différentes époques, nous n’utilisons pas uniquement des totaux bruts de carrière.

Nous construisons d’abord des indicateurs **par pilote et par saison**, ce qui va nous permettre de neutraliser partiellement la longueur des carrières, de comparer des performances dans leur contexte saisonnier, ainsi qu’éviter qu’un pilote moderne soit favorisé uniquement par un plus grand nombre de courses

In [30]:
df_season = (
    df_team
    .groupBy("driverId", "fullName", "year")
    .agg(
        F.count("*").alias("races_count"),
        F.sum("win_flag").alias("wins"),
        F.sum("podium_flag").alias("podiums"),
        F.sum("top10_flag").alias("top10s"),
        F.sum("points").alias("total_points"),
        F.avg("points").alias("points_per_race"),
        F.avg("win_flag").alias("win_rate"),
        F.avg("podium_flag").alias("podium_rate"),
        F.avg("points_flag").alias("points_scoring_rate"),
        F.avg("finish_flag").alias("finish_rate"),
        F.avg("beat_teammate_flag").alias("beat_teammate_rate"),
        F.avg("team_points_share").alias("avg_team_points_share"),
        F.avg("positions_gained").alias("avg_positions_gained")
    )
)

print("✅ Table saison pilote créée")
df_season.orderBy(F.desc("wins")).show(10, truncate=False)

✅ Table saison pilote créée
+--------+------------------+----+-----------+----+-------+------+------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+---------------------+
|driverId|fullName          |year|races_count|wins|podiums|top10s|total_points|points_per_race   |win_rate          |podium_rate       |points_scoring_rate|finish_rate       |beat_teammate_rate|avg_team_points_share|avg_positions_gained |
+--------+------------------+----+-----------+----+-------+------+------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+---------------------+
|830     |Max Verstappen    |2023|22         |19  |21     |22    |530.0       |24.09090909090909 |0.8636363636363636|0.9545454545454546|1.0                |1.0               |0.9090909090909091|0.6975432661218294   |1.9090909090909092   |
|830     |Max Ve

## 7. Classement relatif par saison

Nous créons maintenant un classement relatif des pilotes à l’intérieur de chaque saison.  
L’idée est de ne pas seulement mesurer la performance brute, mais aussi la position d’un pilote par rapport aux autres concurrents de la même année.

In [31]:
w_year_points = Window.partitionBy("year").orderBy(F.col("points_per_race").desc())
w_year_wins = Window.partitionBy("year").orderBy(F.col("win_rate").desc())

In [32]:
df_season_ranked = (
    df_season
    .withColumn("rank_points_per_race", F.dense_rank().over(w_year_points))
    .withColumn("rank_win_rate", F.dense_rank().over(w_year_wins))
)

df_season_ranked.select(
    "year", "fullName", "points_per_race", "rank_points_per_race",
    "win_rate", "rank_win_rate"
).orderBy("year", "rank_points_per_race").show(20, truncate=False)

+----+---------------+------------------+--------------------+-------------------+-------------+
|year|fullName       |points_per_race   |rank_points_per_race|win_rate           |rank_win_rate|
+----+---------------+------------------+--------------------+-------------------+-------------+
|1950|Johnnie Parsons|9.0               |1                   |1.0                |1            |
|1950|Bill Holland   |6.0               |2                   |0.0                |4            |
|1950|Nino Farina    |5.0               |3                   |0.5                |2            |
|1950|Luigi Fagioli  |4.666666666666667 |4                   |0.0                |4            |
|1950|Mauri Rose     |4.0               |5                   |0.0                |4            |
|1950|Juan Fangio    |3.857142857142857 |6                   |0.42857142857142855|3            |
|1950|Cecil Green    |3.0               |7                   |0.0                |4            |
|1950|Dorino Serafini|3.0     

## 8. Normalisation des indicateurs

Avant d’agréger les variables dans un score composite, nous normalisons plusieurs indicateurs sur une échelle comparable.

Nous utilisons ici une normalisation min-max simple afin d’éviter qu’une variable numériquement plus grande domine artificiellement les autres.

In [33]:
agg_stats = df_season.agg(
    F.min("win_rate").alias("min_win_rate"),
    F.max("win_rate").alias("max_win_rate"),
    F.min("podium_rate").alias("min_podium_rate"),
    F.max("podium_rate").alias("max_podium_rate"),
    F.min("points_per_race").alias("min_points_per_race"),
    F.max("points_per_race").alias("max_points_per_race"),
    F.min("beat_teammate_rate").alias("min_beat_teammate_rate"),
    F.max("beat_teammate_rate").alias("max_beat_teammate_rate"),
    F.min("finish_rate").alias("min_finish_rate"),
    F.max("finish_rate").alias("max_finish_rate")
).collect()[0]

agg_stats

Row(min_win_rate=0.0, max_win_rate=1.0, min_podium_rate=0.0, max_podium_rate=1.0, min_points_per_race=0.0, max_points_per_race=24.09090909090909, min_beat_teammate_rate=0.0, max_beat_teammate_rate=1.0, min_finish_rate=0.0, max_finish_rate=1.0)

In [34]:
df_score_base = (
    df_season
    .withColumn(
        "win_rate_norm",
        (F.col("win_rate") - F.lit(agg_stats["min_win_rate"])) /
        (F.lit(agg_stats["max_win_rate"]) - F.lit(agg_stats["min_win_rate"]))
    )
    .withColumn(
        "podium_rate_norm",
        (F.col("podium_rate") - F.lit(agg_stats["min_podium_rate"])) /
        (F.lit(agg_stats["max_podium_rate"]) - F.lit(agg_stats["min_podium_rate"]))
    )
    .withColumn(
        "points_per_race_norm",
        (F.col("points_per_race") - F.lit(agg_stats["min_points_per_race"])) /
        (F.lit(agg_stats["max_points_per_race"]) - F.lit(agg_stats["min_points_per_race"]))
    )
    .withColumn(
        "beat_teammate_rate_norm",
        (F.col("beat_teammate_rate") - F.lit(agg_stats["min_beat_teammate_rate"])) /
        (F.lit(agg_stats["max_beat_teammate_rate"]) - F.lit(agg_stats["min_beat_teammate_rate"]))
    )
    .withColumn(
        "finish_rate_norm",
        (F.col("finish_rate") - F.lit(agg_stats["min_finish_rate"])) /
        (F.lit(agg_stats["max_finish_rate"]) - F.lit(agg_stats["min_finish_rate"]))
    )
)

## 9. Construction du score composite pilote

Nous combinons maintenant plusieurs dimensions de performance :

- taux de victoire
- taux de podium
- points moyens par course
- domination sur le coéquipier
- régularité (taux d’arrivée)


In [35]:
df_score = (
    df_score_base
    .withColumn(
        "composite_score",
        0.30 * F.col("win_rate_norm") +
        0.20 * F.col("podium_rate_norm") +
        0.20 * F.col("points_per_race_norm") +
        0.20 * F.col("beat_teammate_rate_norm") +
        0.10 * F.col("finish_rate_norm")
    )
)

df_score.select(
    "fullName", "year", "composite_score",
    "win_rate", "podium_rate", "points_per_race",
    "beat_teammate_rate", "finish_rate"
).orderBy(F.desc("composite_score")).show(20, truncate=False)

+----------------+----+------------------+------------------+------------------+------------------+------------------+------------------+
|fullName        |year|composite_score   |win_rate          |podium_rate       |points_per_race   |beat_teammate_rate|finish_rate       |
+----------------+----+------------------+------------------+------------------+------------------+------------------+------------------+
|Max Verstappen  |2023|0.9318181818181818|0.8636363636363636|0.9545454545454546|24.09090909090909 |0.9090909090909091|1.0               |
|Lee Wallard     |1951|0.8747169811320755|1.0               |1.0               |9.0               |1.0               |1.0               |
|Johnnie Parsons |1950|0.8747169811320755|1.0               |1.0               |9.0               |1.0               |1.0               |
|Jim Clark       |1968|0.8747169811320755|1.0               |1.0               |9.0               |1.0               |1.0               |
|Bill Vukovich   |1953|0.874716981

In [36]:
top10_drivers = df_score.limit(10)
top10_drivers.show(truncate=False)

+--------+--------------------+----+-----------+----+-------+------+------------+------------------+-------------------+--------------------+-------------------+------------------+-------------------+---------------------+--------------------+-------------------+--------------------+--------------------+-----------------------+------------------+-------------------+
|driverId|fullName            |year|races_count|wins|podiums|top10s|total_points|points_per_race   |win_rate           |podium_rate         |points_scoring_rate|finish_rate       |beat_teammate_rate |avg_team_points_share|avg_positions_gained|win_rate_norm      |podium_rate_norm    |points_per_race_norm|beat_teammate_rate_norm|finish_rate_norm  |composite_score    |
+--------+--------------------+----+-----------+----+-------+------+------------+------------------+-------------------+--------------------+-------------------+------------------+-------------------+---------------------+--------------------+-------------------

## 11. Interprétation des résultats

Le classement obtenu diffère volontairement d’un classement purement basé sur les victoires brutes car il valorise davantage l’efficacité par course, la régularité, la performance relative au coéquipier, la constance saison après saison

Ainsi, un pilote très dominant sur une période plus courte peut être mieux valorisé qu’un pilote ayant simplement accumulé de gros totaux sur une longue carrière.

In [37]:
df_team_check = (
    df_team
    .groupBy("raceId", "constructorId", "constructor_name")
    .agg(
        F.sum("team_points_share").alias("sum_share"),
        F.max("team_total_points_race").alias("team_total_points_race")
    )
    .orderBy("raceId", "constructorId")
)

df_team_check.show(20, truncate=False)

+------+-------------+----------------+---------+----------------------+
|raceId|constructorId|constructor_name|sum_share|team_total_points_race|
+------+-------------+----------------+---------+----------------------+
|1     |1            |McLaren         |0.0      |0.0                   |
|1     |2            |BMW Sauber      |0.0      |0.0                   |
|1     |3            |Williams        |1.0      |3.0                   |
|1     |4            |Renault         |1.0      |4.0                   |
|1     |5            |Toro Rosso      |1.0      |3.0                   |
|1     |6            |Ferrari         |0.0      |0.0                   |
|1     |7            |Toyota          |1.0      |11.0                  |
|1     |9            |Red Bull        |0.0      |0.0                   |
|1     |10           |Force India     |0.0      |0.0                   |
|1     |23           |Brawn           |1.0      |18.0                  |
|2     |1            |McLaren         |1.0      |1.